# 21 · 多头注意力与位置编码

> **本节属于 Part 8 · 注意力与 Transformer。**

单个注意力只能学到一种"关注模式"。**多头注意力**让模型并行地从多个子空间关注信息（有的头看语法、有的头看指代……）。另外，注意力本身**分不清顺序**（打乱输入位置，输出只是跟着打乱）——所以我们要用**位置编码**把"先后"信息注入进去。

## 学习目标

- 实现**多头注意力**：拆分 → 各自注意力 → 拼接
- 理解注意力的"置换等变性"，以及为何需要**位置编码**
- 实现并可视化**正弦位置编码**

## 多头注意力

把 $d_{model}$ 维拆成 $h$ 个 $d_k=d_{model}/h$ 维的"头"，每个头独立做注意力，再把结果拼回来、过一个输出线性层。这样模型能同时学习多种关注方式。

In [ ]:
import inspect
import numpy as np
import matplotlib.pyplot as plt
from minitorch import Tensor, nn
from minitorch.utils import numerical_gradient, rel_error

print(inspect.getsource(nn.MultiheadAttention.forward))

In [ ]:
mha = nn.MultiheadAttention(d_model=32, num_heads=4)
x = Tensor(np.random.randn(2, 6, 32))
out = mha(x)
print("输入 (2,6,32) -> 输出", out.shape)
print("注意力权重形状 (N, heads, L, L):", mha.attn_weights.shape)

# 梯度检查
np.random.seed(0)
mha2 = nn.MultiheadAttention(8, 2); xn = np.random.randn(2, 4, 8); R = np.random.randn(2, 4, 8)
tx = Tensor(xn); (mha2(tx) * Tensor(R)).sum().backward()
g = numerical_gradient(lambda v: float((mha2(Tensor(v)).data * R).sum()), xn.copy())
print("多头注意力对输入梯度 相对误差:", rel_error(tx.grad, g))

## 为什么需要位置编码

注意力对位置是"置换等变"的：如果把输入序列的位置打乱，输出也只是相应打乱——它**看不出谁先谁后**。但语言里顺序至关重要（"狗咬人" ≠ "人咬狗"）。我们来验证这个性质：

In [ ]:
x = np.random.randn(1, 5, 32)        # 维度需与上面的 mha(d_model=32) 一致
perm = [4, 3, 2, 1, 0]
out1 = mha(Tensor(x)).data[0]
out2 = mha(Tensor(x[:, perm, :])).data[0]
print("打乱输入后，输出是否只是同样打乱:", np.allclose(out1[perm], out2, atol=1e-6))
print("=> 注意力本身分不清顺序，必须额外注入位置信息")

## 正弦位置编码

给每个位置一个固定的向量（不同频率的 sin/cos 组合），加到输入上。它能让模型感知绝对位置，也能表达相对距离。

In [ ]:
print(inspect.getsource(nn.PositionalEncoding.forward))

pe = nn.PositionalEncoding(d_model=64, max_len=100)
plt.figure(figsize=(7, 3.2))
plt.imshow(pe.pe[:100].T, cmap="RdBu", aspect="auto"); plt.colorbar()
plt.xlabel("position"); plt.ylabel("encoding dim")
plt.title("Sinusoidal positional encoding"); plt.tight_layout(); plt.show()
print("每一列是一个位置的编码向量；不同维度对应不同频率的波。")

## PyTorch 对照

`nn.MultiheadAttention` 概念一致（PyTorch 的 API 维度约定略有不同，但核心是同一套拆分-注意力-拼接）。位置编码在大多数实现里也是这样加到词嵌入上的。

In [ ]:
import torch
tmha = torch.nn.MultiheadAttention(embed_dim=32, num_heads=4, batch_first=True)
x = torch.randn(2, 6, 32)
out, attn = tmha(x, x, x)
print("PyTorch MHA 输出:", tuple(out.shape), " 注意力:", tuple(attn.shape))
print("minitorch MHA 输出: (2, 6, 32)  注意力: (2, 4, 6, 6)（每个头单独保留）")

## 📦 沉淀进 minitorch

`MultiheadAttention` 在 `minitorch/nn/attention.py`，`PositionalEncoding` 在 `minitorch/nn/transformer.py`，均由测试守护。

## 小练习

1. **头的数量**：固定 `d_model=64`，把 `num_heads` 在 1/2/4/8 间变化，参数量变了吗？（提示：基本不变，只是切分方式不同。）
2. **可学习位置编码**：把正弦编码换成一个可学习的 `Parameter`（随机初始化、随训练更新），它和正弦编码各有什么优劣？
3. **相对位置**：思考为什么正弦编码能隐含"相对距离"信息（提示：sin/cos 的和角公式）。

## 小结 & 下一站

✅ 我们实现了多头注意力与位置编码，理解了"为什么注意力需要被告知位置"。

**下一站 → `22_transformer_block_and_ffn`**：把多头注意力、前馈网络、残差连接与 LayerNorm 组装成一个完整的 **Transformer 编码层**，并叠成可堆叠的编码器。